In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.25 Bell's Inequality and the Failure of Local Realism

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.25",
    title="Bell's Inequality and the Failure of Local Realism",
    blurb="The puzzle of the entangled pair, settled by an experiment. Einstein "
    "hoped the randomness of quantum mechanics hid pre-assigned answers the "
    "particles carried all along — a local, realistic world. Bell turned that "
    "hope into a number: any such world obeys a strict bound on how correlated "
    "two distant measurements can be, and quantum mechanics sails past it. We "
    "build the local model and watch it fail, compute the quantum value that "
    "beats it, and find the sharp limit at which even quantum correlations stop.",
    difficulty="advanced",
    estimate="165–205 min",
)

## Notebook overview

This notebook opens the final movement — foundations — by making rigorous the
puzzle planted in [§6.8](bloch-sphere-entanglement.ipynb). There, the Bell state showed perfect correlations in
incompatible measurement bases while each particle alone looked random, and we
asserted that no local hidden-variable theory could reproduce it. Bell turned
that assertion into a **theorem with a number**.

Einstein, Podolsky, and Rosen believed quantum randomness merely reflected
ignorance of "elements of reality" the particles carry all along — hidden
variables — and that nature is **local** (no instantaneous influence at a
distance). Bell showed this belief is *testable*. Send two entangled spins to
distant detectors, each set to one of two angles; from the correlations form the
**CHSH** combination $S=E(a,b)-E(a,b')+E(a',b)+E(a',b')$. Any theory that is both
**local** and **realistic** must obey $|S|\le2$ — a bound we will *derive* in
three lines and, more vividly, *simulate* by building explicit hidden-variable
models and watching their correlations never beat 2. Quantum mechanics, computed
from the entangled state, gives $|S|=2\sqrt2\approx2.83$ at the optimal geometry:
decisively above the bound. And yet it does not reach the algebraic maximum of 4
either — it stops at **Tsirelson's bound** $2\sqrt2$, more correlated than any
classical theory but strictly less than logic alone would allow.

The distinctive computational move here is to *build the classical model and
watch it fail*: a Monte Carlo of local hidden-variable strategies that never
exceeds 2, set against the quantum value that does. We then simulate a real Bell
test by sampling outcomes from the Born rule ([§6.4](stern-gerlach-qubit.ipynb)), and close with an honest,
evenhanded account of what the violation does and does not establish.

> **Interpretive care.** We separate cleanly what is *demonstrated* — quantum
> mechanics violates a bound that every local-realistic theory obeys, and
> experiments confirm it — from what is *interpreted*, which is genuinely
> contested. The honest conclusion is "local realism fails," not "reality is an
> illusion." We state the assumptions, note the theorem rules out *local* (not
> nonlocal) hidden variables, and present the interpretations without taking a
> side.

> **Method specificity.** The spin operators are $\cos\theta\,\sigma_z+\sin\theta\,
> \sigma_x$; the quantum correlator $\langle\psi|A\otimes B|\psi\rangle$ uses
> `numpy.kron` and `numpy.vdot`; the hidden-variable Monte Carlo and the
> Born-rule sampling use `numpy.random.default_rng`.

## Theory in brief

### The EPR question and hidden variables

The Bell state ([§6.8](bloch-sphere-entanglement.ipynb)) gives perfectly correlated outcomes in several measurement
bases while each particle alone is random. EPR argued this means the outcomes
must be pre-determined by hidden variables the particles carry, and that quantum
mechanics — lacking them — is incomplete; nature should be **local** (a
measurement here cannot instantly affect a distant particle) and **realistic**
(outcomes reflect pre-existing values).

```{math}
:label: eq-epr
\text{EPR: outcomes} = A(\text{setting},\lambda),\ B(\text{setting},\lambda)
\quad\text{for a shared hidden variable } \lambda .
```

### The CHSH setup

Two entangled spins fly to Alice and Bob. Alice measures along one of two angles
($a$ or $a'$), Bob along one of two ($b$ or $b'$); each outcome is $\pm1$. Over
many pairs the **correlation** for a setting pair is $E(a,b)=\langle(\text{Alice})
(\text{Bob})\rangle$, and the CHSH quantity is

```{math}
:label: eq-chsh-setup
S = E(a,b) - E(a,b') + E(a',b) + E(a',b') .
```

### The local-realistic bound (Bell / CHSH)

In any local hidden-variable theory each particle carries pre-assigned outcomes
$A(\cdot,\lambda),B(\cdot,\lambda)=\pm1$, with **locality** (Alice's outcome does
not depend on Bob's setting). Then for each $\lambda$

```{math}
:label: eq-chsh-bound
A(a)[B(b)-B(b')] + A(a')[B(b)+B(b')] = \pm2 ,
```

because one bracket vanishes and the other is $\pm2$. Averaging over $\lambda$
gives $|S|\le2$ — the **CHSH inequality**, a constraint every local-realistic
theory obeys {cite}`bell1964,chsh1969`.

### The quantum prediction

For the entangled (singlet) state the correlation is $E(a,b)=\langle\psi|A\otimes
B|\psi\rangle=-\cos(\theta_a-\theta_b)$. At the optimal geometry $a=0,\ a'=\pi/2,\
b=\pi/4,\ b'=3\pi/4$,

```{math}
:label: eq-chsh-quantum
|S| = 2\sqrt2 \approx 2.83 \;>\; 2 .
```

Quantum correlations are stronger than any local theory permits.

### Tsirelson's bound

Remarkably, quantum mechanics does **not** reach the algebraic maximum of 4. It
stops at $2\sqrt2$ — squaring the CHSH observable leaves $4$ plus a commutator
term whose norm is at most $4$ (Nielsen & Chuang pose the computation as
*Tsirelson's inequality* among the Chapter 2 problems):

```{math}
:label: eq-tsirelson
\text{local-realistic} \le 2 \;<\; \text{quantum} \le 2\sqrt2 \;<\; \text{algebraic } 4 .
```

More correlated than any classical theory, strictly less than logically possible
— a limit that is itself a deep feature of the theory.

### What is demonstrated, and what is interpreted

The **demonstrated** fact is that quantum mechanics violates a bound obeyed by
every local-realistic theory, and that experiments confirm the quantum
prediction (Aspect 1982 {cite}`aspect1982`; the loophole-free tests of 2015
{cite}`hensen2015`; the 2022 Nobel Prize to Aspect, Clauser, and Zeilinger).

```{math}
:label: eq-interpretation
\text{violation} \Rightarrow \{\text{locality, realism, measurement independence}\}
\text{ cannot all hold.}
```

What this **means** needs care. The theorem rules out *local* hidden variables
but not *nonlocal* ones (Bohmian mechanics reproduces quantum mechanics by being
explicitly nonlocal), and it does not permit faster-than-light signalling — the
marginals stay random (the no-signalling theorem). The interpretations
(Copenhagen, many-worlds, Bohmian, and others) accept different horns; we present
them without adjudicating. The honest conclusion is "local realism fails," not
"reality is an illusion."

- Reference: Bell {cite}`bell1964`; CHSH {cite}`chsh1969`; Aspect et al.
  {cite}`aspect1982`; the loophole-free experiments {cite}`hensen2015`; Nielsen &
  Chuang {cite}`nielsen_chuang`. Cross-reference [§6.8](bloch-sphere-entanglement.ipynb) (the Bell state,
  entanglement, the puzzle posed), [§6.6](pauli-uncertainty.ipynb) (incompatible observables), [§6.5](postulates.ipynb) (the Born
  rule), [§6.4](stern-gerlach-qubit.ipynb) (Born-rule Monte Carlo), and forward to [§6.26](density-matrix.ipynb) (the density matrix),
  [§6.27](quantum-information.ipynb) (quantum information). Named as horizons: GHZ states, the detection and
  locality loopholes, device-independent quantum cryptography.

---
## Setup

We use the qubit form of CHSH: two spin-$\tfrac12$ particles in the **singlet**
state $|\psi^-\rangle=(|01\rangle-|10\rangle)/\sqrt2$, each measured along an axis
in the $x$–$z$ plane at angle $\theta$. Outcomes are the $\pm1$ eigenvalues of the
spin operator. Conventions: angles in radians; Alice's qubit is the first factor.
The reusable core:

- `spin_operator(theta)`: the observable $\cos\theta\,\sigma_z+\sin\theta\,\sigma_x$.
- `correlation(tA, tB, state)`: the quantum $E(a,b)=\langle\psi|A\otimes B|\psi
  \rangle$ via `numpy.kron` and `numpy.vdot`.
- `chsh(state, a, ap, b, bp)`: the CHSH combination {eq}`eq-chsh-setup`.
- `lhv_chsh(rng, ...)`: a simulated local hidden-variable CHSH value.
- `born_sampled_correlation(tA, tB, n, rng)`: a correlation estimated by sampling
  outcomes from the Born rule ([§6.4](stern-gerlach-qubit.ipynb)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ecp import draw, validate

SIGMA_X = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
SIGMA_Z = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)
ID2 = np.eye(2, dtype=complex)

# The singlet (Bell) state |ψ⁻⟩ = (|01⟩ − |10⟩)/√2, Alice's qubit first.
SINGLET = (np.kron([1.0, 0.0], [0.0, 1.0]) - np.kron([0.0, 1.0], [1.0, 0.0])) / np.sqrt(
    2
)

# The four optimal CHSH angles.
OPTIMAL = dict(a=0.0, ap=np.pi / 2, b=np.pi / 4, bp=3 * np.pi / 4)


def spin_operator(theta):
    r"""Spin observable along angle ``theta`` in the $x$–$z$ plane.

    Returns $A(\theta)=\cos\theta\,\sigma_z+\sin\theta\,\sigma_x$, whose $\pm1$
    eigenvalues are the two measurement outcomes.

    Parameters
    ----------
    theta : float
        Measurement angle (radians).

    Returns
    -------
    numpy.ndarray
        The $2\times2$ Hermitian spin operator.
    """
    return np.cos(theta) * SIGMA_Z + np.sin(theta) * SIGMA_X


def correlation(tA, tB, state=SINGLET):
    r"""Quantum correlation $E(a,b)=\langle\psi|A(\theta_a)\otimes B(\theta_b)|\psi\rangle$.

    The joint observable is `numpy.kron(A, B)`; the expectation is
    `numpy.vdot(state, op @ state)` (real for a Hermitian operator).

    Parameters
    ----------
    tA, tB : float
        Alice's and Bob's measurement angles.
    state : numpy.ndarray, optional
        The two-qubit state (default the singlet).

    Returns
    -------
    float
        The correlation $E(a,b)\in[-1,1]$.
    """
    op = np.kron(spin_operator(tA), spin_operator(tB))
    return float(np.real(np.vdot(state, op @ state)))


def chsh(state, a, ap, b, bp):
    r"""The CHSH combination $S=E(a,b)-E(a,b')+E(a',b)+E(a',b')$ {eq}`eq-chsh-setup`."""
    return (
        correlation(a, b, state)
        - correlation(a, bp, state)
        + correlation(ap, b, state)
        + correlation(ap, bp, state)
    )


def _outcome_bases(theta):
    r"""Return the $(+1, -1)$ eigenvectors of ``spin_operator(theta)``."""
    w, v = np.linalg.eigh(spin_operator(theta))  # ascending: w = (-1, +1)
    return v[:, 1], v[:, 0]  # (+1 vector, -1 vector)


def born_sampled_correlation(tA, tB, n, rng, state=SINGLET):
    r"""Correlation estimated by Born-rule sampling of outcomes (§6.4).

    Builds the four joint-outcome probabilities $|\langle a_\pm b_\pm|\psi\rangle|^2$
    and draws ``n`` outcome pairs with `numpy.random.default_rng`, returning the
    sample mean of the outcome product — a simulated Bell-test correlation.

    Parameters
    ----------
    tA, tB : float
        Measurement angles.
    n : int
        Number of pairs sampled.
    rng : numpy.random.Generator
        Random generator.
    state : numpy.ndarray, optional
        Two-qubit state.

    Returns
    -------
    float
        The sampled correlation.
    """
    aP, aM = _outcome_bases(tA)
    bP, bM = _outcome_bases(tB)
    outs, probs = [], []
    for sa, va in ((+1, aP), (-1, aM)):
        for sb, vb in ((+1, bP), (-1, bM)):
            probs.append(abs(np.vdot(np.kron(va, vb), state)) ** 2)
            outs.append(sa * sb)
    probs = np.array(probs)
    probs /= probs.sum()
    draws = rng.choice(len(outs), size=n, p=probs)
    return float(np.mean(np.array(outs)[draws]))

## Exercise 1 — The entangled pair and its correlations

For the singlet state, compute the correlation $E(a,b)$ between spin measurements along two
angles, and show it is $-\cos(\theta_a-\theta_b)$ — the correlations Einstein thought demanded
hidden variables.

1. Build the singlet $|\psi^-\rangle=(|01\rangle-|10\rangle)/\sqrt2$ ([§6.8](bloch-sphere-entanglement.ipynb)).
2. Build the spin operators $A(\theta_a),B(\theta_b)$ along the two angles.
3. Compute $E(a,b)=\langle\psi|A\otimes B|\psi\rangle$ with `numpy.kron`/`numpy.vdot`.
4. Confirm $E(a,b)=-\cos(\theta_a-\theta_b)$: perfect anticorrelation at equal
   angles, and the smooth cosine in between.

Cite {eq}`eq-epr`, {eq}`eq-chsh-setup`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

The singlet correlation must equal $-\cos(\theta_a-\theta_b)$ for every pair of
angles: anticorrelated at equal settings, and the cosine law in between.

In [ ]:
angles = [(0.0, 0.0), (0.0, np.pi / 4), (np.pi / 3, np.pi / 7), (1.1, -0.4)]
validate.close(
    np.array([correlation(tA, tB) for tA, tB in angles]),
    np.array([-np.cos(tA - tB) for tA, tB in angles]),
    "the singlet correlation is E(a,b) = −cos(θ_a − θ_b)",
    rtol=1e-9,
    atol=1e-9,
)

## Exercise 2 — Deriving the CHSH bound for local realism

Show that any local hidden-variable theory obeys $|S|\le2$.

1. Write the pre-assigned outcomes $A(a,\lambda),A(a',\lambda),B(b,\lambda),
   B(b',\lambda)=\pm1$ (locality: $A$ independent of Bob's setting).
2. Form $S(\lambda)=A(a)[B(b)-B(b')]+A(a')[B(b)+B(b')]$.
3. Note that $B(b)$ and $B(b')$ are each $\pm1$, so one bracket is $0$ and the
   other is $\pm2$; hence $S(\lambda)=\pm2$ for *every* $\lambda$.
4. Average over $\lambda$: $|\langle S\rangle|\le2$.

We verify step 3 by brute force over all $2^4$ sign assignments.

Cite {eq}`eq-chsh-bound`.

In [ ]:
# (solution hidden on the public site)


### Validation 2

For every pre-assignment of outcomes, $S(\lambda)=\pm2$; therefore any average
over $\lambda$ satisfies the CHSH bound $|S|\le2$.

In [ ]:
validate.check(
    np.all(np.abs(S_lambda) == 2),
    "S(λ) = ±2 for every hidden-variable assignment, so any local-realistic "
    "theory obeys the CHSH bound |S| ≤ 2",
)

## Exercise 3 — Simulating a local hidden-variable model

Build explicit local hidden-variable models and Monte Carlo their CHSH value; confirm that no
strategy exceeds 2, however hard it tries. This is the distinctive move — build the classical
model and watch it fail.

1. Draw a shared hidden variable $\lambda$ per pair with `numpy.random.default_rng`.
2. Define **local** deterministic outcomes $A(\text{setting},\lambda),
   B(\text{setting},\lambda)=\pm1$ — Alice's rule cannot see Bob's setting.
3. Estimate the four correlations by averaging over many $\lambda$, and form $S$.
4. Repeat over hundreds of random strategies and confirm $|S|$ never beats 2 (up
   to Monte Carlo noise).

Cite {eq}`eq-chsh-bound`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

Over hundreds of local hidden-variable strategies, the simulated CHSH value must
stay at or below 2 (within Monte Carlo noise) — no local model exceeds the bound.

In [ ]:
validate.check(
    lhv_values.max() <= 2.02,
    "the simulated local hidden-variable CHSH value stays ≤ 2 over hundreds of "
    "strategies — no local model exceeds the CHSH bound",
)

## Exercise 4 — The quantum violation

Compute the CHSH value for the singlet at the optimal measurement angles and show
$|S|=2\sqrt2\approx2.83>2$ — a violation of the local-realistic bound.

1. Choose $a=0,\ a'=\pi/2,\ b=\pi/4,\ b'=3\pi/4$.
2. Compute the four quantum correlations from the singlet.
3. Form $S$ and take its magnitude.
4. Confirm $|S|=2\sqrt2$ — beyond every local theory. (The singlet is
   anticorrelated, so $S=-2\sqrt2$; the physical statement is the CHSH one,
   $|S|\le2$, which quantum mechanics breaks.)

Cite {eq}`eq-chsh-quantum`.

In [ ]:
# (solution hidden on the public site)


### Validation 4

At the optimal geometry the quantum CHSH magnitude must equal $2\sqrt2$,
decisively above the local-realistic bound of 2.

In [ ]:
validate.close(
    abs(S_quantum),
    2 * np.sqrt(2),
    "quantum mechanics gives |S| = 2√2 ≈ 2.83, violating the local-realistic bound of 2",
    rtol=1e-6,
)

## Exercise 5 — The angle scan

Map the quantum CHSH value over measurement geometries and show it exceeds the classical bound
over a broad range, peaking at $2\sqrt2$.

1. Parametrize the four angles by one geometry parameter: $a=0,\ a'=2\varphi,\
   b=\varphi,\ b'=3\varphi$.
2. Compute $|S(\varphi)|$ over a scan of $\varphi$.
3. Show the peak is $2\sqrt2$ at the optimal geometry ($\varphi=\pi/4$, the
   equally-spaced $22.5^\circ$ arrangement).
4. Note $|S|>2$ over most of the scan — the violation is robust, not fine-tuned.

Cite {eq}`eq-chsh-quantum`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

The scanned quantum CHSH value must peak at $2\sqrt2$ and exceed 2 over a broad
range of geometries (not a single fine-tuned point).

In [ ]:
validate.close(
    S_scan.max(),
    2 * np.sqrt(2),
    "the CHSH value peaks at 2√2 for the optimal geometry and exceeds 2 broadly",
    rtol=1e-3,
)
validate.check(
    frac_above > 0.5, "the quantum violation |S|>2 holds over most of the scan (robust)"
)

## Exercise 6 — Tsirelson's bound

Show that quantum mechanics cannot exceed $2\sqrt2$, even though the algebraic maximum of $S$ is
4.

1. Note the algebraic maximum: if all four correlations were $\pm1$, $|S|$ could
   reach 4.
2. Show the quantum scan never exceeds $2\sqrt2$.
3. State the ordering: local-realistic $\le2<$ quantum $\le2\sqrt2<$ algebraic 4.
4. Remark that $2\sqrt2$ is a genuine feature of quantum mechanics — *why* exactly
   $2\sqrt2$ is still a research question.

Cite {eq}`eq-tsirelson`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

The quantum CHSH value must never exceed $2\sqrt2$: quantum mechanics respects
Tsirelson's bound, sitting strictly between the classical bound and the algebraic
maximum.

In [ ]:
validate.check(
    S_scan.max() <= 2 * np.sqrt(2) + 1e-6,
    "the quantum CHSH value never exceeds 2√2 (Tsirelson): local ≤2 < quantum ≤2√2 < 4",
)

## Exercise 7 — Simulating the actual experiment *(student)*

Simulate a real Bell test by sampling measurement outcomes from the Born rule and estimate $S$
from finite data, as an actual experiment does.

1. For each setting pair, compute the joint outcome probabilities from the singlet.
2. Sample outcomes with `numpy.random.default_rng` (the Born-rule Monte Carlo of
   [§6.4](stern-gerlach-qubit.ipynb)) via `born_sampled_correlation`.
3. Estimate the four correlations from the samples and form $S$.
4. Confirm $|S|\approx2\sqrt2$ within statistical error — this is what the
   Nobel-winning experiments measured.

Cite {eq}`eq-chsh-quantum`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

Born-rule sampling of the entangled state must reproduce the CHSH violation:
$|S|\approx2\sqrt2$ within statistical error, and comfortably above 2.

In [ ]:
validate.close(
    S_sampled,
    2 * np.sqrt(2),
    "Born-rule sampling of the entangled state reproduces the CHSH violation (≈2√2)",
    atol=0.05,
)

## Exercise 8 — What must give way *(student / interpretation)*

Articulate precisely what the violation does and does not establish, and verify the no-signalling
property that keeps it consistent with relativity.

1. List Bell's three assumptions: **locality** (an outcome depends only on the
   local setting and shared past), **realism** (outcomes are determined by
   pre-existing values), and **measurement independence** (settings are chosen
   freely, independently of $\lambda$).
2. State that the violation means at least one must fail — it rules out *local*
   hidden variables, but not *nonlocal* ones (Bohmian mechanics is explicitly
   nonlocal and reproduces quantum mechanics).
3. Verify **no-signalling**: Alice's marginal outcome probability is independent
   of Bob's setting, so no information is transmitted — the violation does not
   permit faster-than-light communication.
4. Summarize the interpretation landscape (Copenhagen, many-worlds, Bohmian, and
   others) without choosing. The honest conclusion is "local realism fails," not
   "reality is an illusion."

Cite {eq}`eq-interpretation`.

In [ ]:
# (solution hidden on the public site)


### Validation 8

Alice's single-party marginal must be independent of Bob's distant setting
(always $1/2$): the violation defeats local realism but forbids signalling — the
marginals carry no information.

In [ ]:
validate.close(
    np.array(marginals),
    0.5 * np.ones(4),
    "the single-party marginals are 1/2 regardless of the distant setting "
    "(no signalling: the violation forbids local realism, not relativity)",
    atol=1e-9,
)

## Exercise 9 — (Synthesis) A number that decided a debate

For thirty years the question of whether quantum randomness hid a deeper, local,
deterministic layer was thought to be metaphysics, beyond experiment. Bell turned
it into arithmetic: a single number, bounded by 2 for any local-realistic world,
predicted to be $2\sqrt2$ by quantum mechanics. We built the local model and
watched it strain against the bound and fail — no strategy, however clever, beat
2. We computed the quantum value and watched it pass, at $2\sqrt2\approx2.83$. And
we found that quantum mechanics, for all its strangeness, stops short of the
maximum imaginable: more correlated than any classical theory, less than logic
alone would allow. Born-rule sampling reproduced the violation from simulated
coincidence counts, exactly as a real experiment does.

It is worth pausing on how unusual this is. A philosophical dispute between
Einstein and Bohr about the nature of reality was settled — not dissolved,
settled — by a table of measured coincidence counts. Aspect's 1982 experiment,
the loophole-free tests of 2015, and the 2022 Nobel Prize confirm it: the
correlations of the Bell state are real, they defeat local realism, and no
experiment has ever sided against them. What remains open is not *whether*
quantum mechanics is right but *what it means* — and on that, the honest word is
that at least one of locality, realism, or free choice must go, while the
no-signalling theorem keeps the whole thing consistent with relativity. Which
assumption to surrender is where Copenhagen, many-worlds, and Bohmian mechanics
part ways, and this notebook takes no side.

With that, **Movement VI is open** — the foundations, where we ask not how to
compute with quantum mechanics but what it is telling us. The next notebook ([§6.26](density-matrix.ipynb))
steps back to ask what "the state of a subsystem" even means when parts are
entangled like this: the **density matrix**, the proper description of the mixed,
correlated pieces that Bell pairs are made of.

## Notebook summary

- **The singlet correlations** {eq}`eq-chsh-setup`: $E(a,b)=-\cos(\theta_a-
  \theta_b)$, computed as $\langle\psi|A\otimes B|\psi\rangle$ (`numpy.kron`/`vdot`).
- **The CHSH bound** {eq}`eq-chsh-bound`: $S(\lambda)=\pm2$ for every
  pre-assignment, so any local-realistic theory obeys $|S|\le2$ (derived + brute-checked).
- **The local model fails**: hundreds of simulated hidden-variable strategies
  (`numpy.random.default_rng`) never beat 2.
- **The quantum violation** {eq}`eq-chsh-quantum`: $|S|=2\sqrt2\approx2.83$ at the
  optimal angles, exceeding 2 over ~76% of geometries.
- **Tsirelson** {eq}`eq-tsirelson`: quantum $|S|\le2\sqrt2<4$ — strong but not maximal.
- **The experiment, simulated**: Born-rule sampling reproduces $|S|\approx2\sqrt2$;
  the marginals stay $1/2$ (no signalling). The honest conclusion: **local realism
  fails**.

> **What is settled, and what is not.** *Settled:* quantum mechanics violates a
> bound every local-realistic theory obeys, and experiment agrees. *Open:* which
> assumption to give up, and what the violation means. This notebook demonstrates
> the first and lays out the second without adjudicating.

## Outlook

- **The density matrix and mixed states** ([§6.26](density-matrix.ipynb)): the proper description of a
  subsystem of an entangled pair.
- **Quantum information and device-independent cryptography** ([§6.27](quantum-information.ipynb)), where Bell
  violations *certify* security.
- **GHZ states**: a single-measurement, all-or-nothing version of the
  contradiction (a horizon).
- **The loopholes** (detection, locality, freedom-of-choice) and the loophole-free
  experiments (a horizon).
- Cross-reference [§6.8](bloch-sphere-entanglement.ipynb) (the Bell state, entanglement), [§6.6](pauli-uncertainty.ipynb) (incompatible
  observables), [§6.5](postulates.ipynb) (the Born rule), [§6.4](stern-gerlach-qubit.ipynb) (Born-rule Monte Carlo), forward to [§6.26](density-matrix.ipynb),
  [§6.27](quantum-information.ipynb).

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()